<a href="https://colab.research.google.com/github/Erjg1012/big-data-con-Spark-/blob/main/ETL_spark_ds_estructurado.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F


In [ ]:


# ============================================================
# 1. EXTRACT
# ============================================================

def extract(spark: SparkSession, file_path: str):
    """
    Extrae datos desde un CSV usando Spark y muestra información básica.
    """
    df = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "true")
        .csv(file_path)
    )

    print("\n" + "=" * 60)
    print(f"[EXTRACT] Archivo: {file_path}")
    print(f"[EXTRACT] Filas (approx): {df.count()}, Columnas: {len(df.columns)}")
    print("[EXTRACT] Esquema:")
    df.printSchema()
    print("=" * 60)

    return df


# ============================================================
# 2. TRANSFORM
# ============================================================

def transform(apps_df, reviews_df, category: str, min_rating: float, min_reviews: int):
    """
    Aplica la misma lógica de negocio que la versión con pandas, pero en Spark:

    - elimina duplicados
    - filtra apps por categoría
    - se queda con reviews de esas apps
    - calcula promedio de Sentiment_Polarity por App
    - hace join con apps
    - selecciona columnas relevantes
    - convierte Reviews a entero
    - filtra por min_rating y min_reviews
    - ordena por Reviews desc
    """

    print("\n" + "=" * 60)
    print(f"[TRANSFORM] Categoría objetivo: {category}")
    print(f"[TRANSFORM] Mínimo rating: {min_rating}")
    print(f"[TRANSFORM] Mínimo reviews: {min_reviews}")
    print("=" * 60)

    # 1) Eliminar duplicados
    reviews_clean = reviews_df.dropDuplicates()
    apps_clean = apps_df.dropDuplicates(["App"])

    # 2) Filtrar apps por categoría
    subset_apps = apps_clean.filter(F.col("Category") == category)

    # 3) Quedarse solo con reviews de esas apps
    #    (join inner con la lista de apps filtradas)
    subset_reviews = (
        reviews_clean
        .join(
            subset_apps.select("App").distinct(),
            on="App",
            how="inner"
        )
        .select("App", "Sentiment_Polarity")
    )

    # 4) Calcular promedio de Sentiment_Polarity por App

    ##########################################################################################################################3
    print("\n" + "*" * 60)
    print("[TRANSFORM] Subset de reviews antes de limpiar:")
    subset_reviews.printSchema()
    subset_reviews.show(5, truncate=False)

    # 4.1 Extraer solo el número de la columna Sentiment_Polarity (que viene como string sucio)
    subset_reviews = subset_reviews.withColumn(
        "Sentiment_Clean",
        F.regexp_extract(F.col("Sentiment_Polarity"), r"[-+]?\d*\.?\d+", 0)
    )

    # 4.2 Convertir ese número limpio a double usando try_cast
    subset_reviews = subset_reviews.withColumn(
        "Sentiment_Clean",
        F.expr("try_cast(Sentiment_Clean AS double)")
    )

    # 4.3 Eliminar filas donde no se pudo obtener un número (NULL o NaN)
    subset_reviews = subset_reviews.where(
        ~F.isnull(F.col("Sentiment_Clean")) & ~F.isnan(F.col("Sentiment_Clean"))
    )

    print("[TRANSFORM] Subset de reviews después de limpiar:")
    subset_reviews.printSchema()
    subset_reviews.show(5, truncate=False)
    print("\n" + "*" * 60)

    # 4.4 Calcular promedio de Sentiment_Clean por App
    aggregated_reviews = (
        subset_reviews
        .groupBy("App")
        .agg(F.avg("Sentiment_Clean").alias("Sentiment_Polarity"))
    )

    # 5) Join left entre apps filtradas y reviews agregadas
    merged = subset_apps.join(aggregated_reviews, on="App", how="left")

    # 6) Seleccionar columnas de interés
    filtered = merged.select("App", "Rating", "Reviews", "Size", "Sentiment_Polarity")

    # 7) Convertir Reviews a entero (o null si no se puede)
    filtered = filtered.withColumn(
        "Reviews",
        F.col("Reviews").cast("int")
    )

    # 8) Aplicar filtros de negocio
    filtered = filtered.filter(
        (F.col("Rating") >= F.lit(min_rating)) &
        (F.col("Reviews") >= F.lit(min_reviews))
    )

    # 9) Ordenar por número de reviews (descendiente)
    filtered = filtered.orderBy(F.col("Reviews").desc())

    # (Opcional) Mostrar algunas filas
    print("[TRANSFORM] Vista previa del resultado:")
    print("\n" + "*" * 60)
    filtered.show(10, truncate=False)

    print("[TRANSFORM] Transformación completa.")
    print("=" * 60)

    return filtered


# ============================================================
# 3. LOAD
# ============================================================

def load(df, output_path: str, spark: SparkSession):
    """
    Carga el DataFrame resultante a almacenamiento en formato Parquet.
    (En un entorno real sería HDFS, S3, Data Lake, etc.)
    """

    print("\n" + "=" * 60)
    print(f"[LOAD] Guardando resultado en: {output_path}")
    (
        df
        .coalesce(1)          # opcional: para tener un solo archivo de salida
        .write
        .mode("overwrite")
        .parquet(output_path)
    )
    print("[LOAD] Escritura completada en formato Parquet.")
    print("=" * 60)

    df_loaded = spark.read.parquet("output/top_apps_parquet")
    df_loaded.show(60, truncate=False)



[EXTRACT] Archivo: googleplaystore.csv
[EXTRACT] Filas (approx): 10841, Columnas: 13
[EXTRACT] Esquema:
root
 |-- App: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Rating: string (nullable = true)
 |-- Reviews: string (nullable = true)
 |-- Size: string (nullable = true)
 |-- Installs: string (nullable = true)
 |-- Type: string (nullable = true)
 |-- Price: string (nullable = true)
 |-- Content Rating: string (nullable = true)
 |-- Genres: string (nullable = true)
 |-- Last Updated: string (nullable = true)
 |-- Current Ver: string (nullable = true)
 |-- Android Ver: string (nullable = true)


[EXTRACT] Archivo: googleplaystore_user_reviews.csv
[EXTRACT] Filas (approx): 64295, Columnas: 5
[EXTRACT] Esquema:
root
 |-- App: string (nullable = true)
 |-- Translated_Review: string (nullable = true)
 |-- Sentiment: string (nullable = true)
 |-- Sentiment_Polarity: string (nullable = true)
 |-- Sentiment_Subjectivity: string (nullable = true)


[TRANSFORM] Categoría

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window


def transform_top_free_by_installs(apps_df, n_per_bucket=5):


    df = apps_df.filter(F.col("Type") == "Free")


    df = df.withColumn(
        "Installs_Clean",
        F.regexp_replace(F.col("Installs"), "[+,]", "").cast("int")
    )

    df = df.withColumn(
        "Installs_Bucket",
        F.when(F.col("Installs_Clean") < 10_000, "0-10k") \
         .when(F.col("Installs_Clean") < 100_000, "10k-100k") \
         .when(F.col("Installs_Clean") < 1_000_000, "100k-1M") \
         .otherwise("1M+")
    )

    w = Window.partitionBy("Installs_Bucket").orderBy(F.col("Reviews").cast("int").desc())

    df = df.withColumn("rank_in_bucket", F.row_number().over(w))

    df_top = df.filter(F.col("rank_in_bucket") <= n_per_bucket)

    result = df_top.select(
        "App",
        "Category",
        "Rating",
        "Reviews",
        "Installs",
        "Installs_Bucket",
        "rank_in_bucket"
    ).orderBy("Installs_Bucket", "rank_in_bucket")

    return result


def transform_sentiment_by_category(apps_df, reviews_df):

    joined = (
        reviews_df
        .join(apps_df.select("App", "Category"), on="App", how="inner")
    )

    joined = joined.withColumn(
        "Sentiment_Clean",
        F.expr("try_cast(Sentiment_Polarity AS double)")
    )

    joined = joined.withColumn(
        "Sentiment_Label",
        F.when(F.col("Sentiment_Clean") <= -0.1, "negative")
         .when(F.col("Sentiment_Clean") >= 0.1, "positive")
         .otherwise("neutral")
    )

    grouped = (
        joined
        .groupBy("Category", "Sentiment_Label")
        .agg(
            F.count("*").alias("count_label"),
            F.avg("Sentiment_Clean").alias("avg_polarity_label")
        )
    )

    total_per_cat = (
        joined.groupBy("Category")
        .agg(
            F.count("*").alias("total_reviews"),
            F.avg("Sentiment_Clean").alias("avg_polarity_category")
        )
    )

    pivoted = (
        grouped
        .groupBy("Category")
        .pivot("Sentiment_Label", ["negative", "neutral", "positive"])
        .agg(F.first("count_label").alias("count"))
    )

    result = (
        total_per_cat
        .join(pivoted, on="Category", how="left")
        .orderBy("total_reviews", ascending=False)
    )

    return result


def transform_sentiment_by_category(apps_df, reviews_df):

    print("\n" + "=" * 60)
    print("[TRANSFORM 2] Sentiment por categoría")
    print("=" * 60)

    joined = (
        reviews_df.alias("r")
        .join(
            apps_df.select("App", "Category").alias("a"),
            on="App",
            how="inner"
        )
    )

    print("[DEBUG] Esquema después del join:")
    joined.printSchema()
    joined.select("App", "Category", "Sentiment", "Sentiment_Polarity") \
          .show(5, truncate=False)

    joined = joined.withColumn(
        "SentPol_str",
        F.col("Sentiment_Polarity").cast("string")
    )

    joined = joined.withColumn(
        "SentPol_num_str",
        F.regexp_extract(F.col("SentPol_str"), r"[-+]?\d*\.?\d+", 0)
    )

    joined = joined.withColumn(
        "SentPol_num",
        F.expr("try_cast(SentPol_num_str AS double)")
    )

    joined.select(
        F.count("*").alias("total_reviews"),
        F.count("SentPol_num").alias("con_polaridad_valida")
    ).show()

    print("[DEBUG] Ejemplos de polaridad limpia:")
    joined.select(
        "App", "Sentiment", "Sentiment_Polarity", "SentPol_num"
    ).show(10, truncate=False)

    result = (
        joined.groupBy("Category")
        .agg(
            F.count("*").alias("total_reviews"),
            F.avg("SentPol_num").alias("avg_polarity_category"),
            F.sum(F.when(F.col("Sentiment") == "Negative", 1).otherwise(0)).alias("negative"),
            F.sum(F.when(F.col("Sentiment") == "Neutral", 1).otherwise(0)).alias("neutral"),
            F.sum(F.when(F.col("Sentiment") == "Positive", 1).otherwise(0)).alias("positive"),
        )
        .orderBy(F.col("total_reviews").desc())
    )

    print("[TRANSFORM 2] Resultado (primeras filas):")
    result.show(60, truncate=False)

    return result


from pyspark.sql import functions as F

def transform_recent_quality_apps(apps_df, ref_date_str="2018-01-01"):

    print("\n" + "=" * 60)
    print("[TRANSFORM 4] Apps recientes y calidad")
    print("=" * 60)

    df = apps_df.withColumn("LastUpdated_str", F.col("Last Updated").cast("string"))

    df = df.withColumn(
        "LastUpdated_ts",
        F.expr("try_to_timestamp(LastUpdated_str, 'MMMM d, yyyy')")
    )

    print("[DEBUG] Ejemplos de Last Updated y parseo:")
    df.select("App", "LastUpdated_str", "LastUpdated_ts") \
      .show(10, truncate=False)

    ref_ts = F.to_timestamp(F.lit(ref_date_str), "yyyy-MM-dd")

    df = df.withColumn(
        "is_recent",
        (F.col("LastUpdated_ts") >= ref_ts)
    )

    df = df.withColumn("Rating_str", F.col("Rating").cast("string"))
    df = df.withColumn(
        "Rating_num_str",
        F.regexp_extract(F.col("Rating_str"), r"[-+]?\d*\.?\d+", 0)
    )
    df = df.withColumn(
        "Rating_num",
        F.expr("try_cast(Rating_num_str AS double)")
    )

    print("[DEBUG] Ejemplos de Rating limpio:")
    df.select("App", "Rating", "Rating_num").show(10, truncate=False)

    df = df.withColumn(
        "quality_badge",
        F.when(F.col("Rating_num") >= 4.5, "TOP")
         .when(F.col("Rating_num") >= 4.0, "OK")
         .when(F.col("Rating_num").isNotNull(), "LOW")
         .otherwise("NO_RATING")
    )

    df = df.withColumn(
        "is_recent_and_top",
        (F.col("is_recent") & (F.col("quality_badge") == "TOP"))
    )

    result = (
        df.select(
            "App",
            "Category",
            "Rating_num",
            "LastUpdated_ts",
            "is_recent",
            "quality_badge",
            "is_recent_and_top"
        )
        .orderBy(F.col("LastUpdated_ts").desc_nulls_last(), F.col("Rating_num").desc_nulls_last())
    )

    print("[TRANSFORM 4] Resultado (primeras filas):")
    result.show(20, truncate=False)

    return result



def main():
    # Crear SparkSession
    spark = (
        SparkSession.builder
        .appName("ETL_GooglePlay_Spark")
        .master("local[*]")   # local, usando todos los núcleos disponibles
        .getOrCreate()
    )

    # Rutas a los CSV
    apps_csv_path = "googleplaystore.csv"
    reviews_csv_path = "googleplaystore_user_reviews.csv"


    # Parámetros de negocio
    target_category = "FOOD_AND_DRINK"
    min_rating = 4.0
    min_reviews = 1000

    # Ruta de salida (Parquet)
    output_path = "output/top_apps_parquet"

    # 1) EXTRACT
    apps_df = extract(spark, apps_csv_path)
    reviews_df = extract(spark, reviews_csv_path)

    # 2) TRANSFORM

    # top_apps_df = transform(
    #     apps_df=apps_df,
    #     reviews_df=reviews_df,
    #     category=target_category,
    #     min_rating=min_rating,
    #     min_reviews=min_reviews
    # )

    top_apps_df = transform_top_free_by_installs(apps_df
        ,n_per_bucket=5
    )

    #top_apps_df = transform_sentiment_by_category(
     #   apps_df=apps_df,
     #   reviews_df=reviews_df
    #)

    #top_apps_df = transform_genres_exploded(
    #    apps_df=apps_df,
        #reviews_df=reviews_df
    #)

    #top_apps_df = transform_recent_quality_apps(
    #    apps_df=apps_df,
        #reviews_df=reviews_df
    #)


    # 3) LOAD
    load(top_apps_df, output_path=output_path, spark=spark)



    # Cerrar sesión de Spark
    spark.stop()



main()



[EXTRACT] Archivo: googleplaystore.csv
[EXTRACT] Filas (approx): 10841, Columnas: 13
[EXTRACT] Esquema:
root
 |-- App: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Rating: string (nullable = true)
 |-- Reviews: string (nullable = true)
 |-- Size: string (nullable = true)
 |-- Installs: string (nullable = true)
 |-- Type: string (nullable = true)
 |-- Price: string (nullable = true)
 |-- Content Rating: string (nullable = true)
 |-- Genres: string (nullable = true)
 |-- Last Updated: string (nullable = true)
 |-- Current Ver: string (nullable = true)
 |-- Android Ver: string (nullable = true)


[EXTRACT] Archivo: googleplaystore_user_reviews.csv
[EXTRACT] Filas (approx): 64295, Columnas: 5
[EXTRACT] Esquema:
root
 |-- App: string (nullable = true)
 |-- Translated_Review: string (nullable = true)
 |-- Sentiment: string (nullable = true)
 |-- Sentiment_Polarity: string (nullable = true)
 |-- Sentiment_Subjectivity: string (nullable = true)


[LOAD] Guardando resu